# LLM Log Triage Explorer (EDD workflow)

Interactive **evaluation-driven development** for notebook users. All live calls go through `llm_log_triage.notebook_runner` → **`interface:notebook`** in LangSmith + JSON report in [`docs/eval-runs/notebook-latest.json`](../docs/eval-runs/notebook-latest.json).

**EDD loop (this notebook):**
1. Pick a case from `data/golden_set.json`
2. Run triage → structured JSON
3. Score vs expected labels (deterministic checks)
4. Optional: LLM-as-Judge on one case
5. If fail → inspect here; if good → pytest remains the automation gate

Full CI gate: `pytest tests/ -m llm -v` and `pytest tests/ -m judge -v`

## How to run

1. Kernel → **`.venv` / Python 3.12`** (top-right picker in Cursor)
2. Run the **Setup** cell below first (resets session + loads golden set; free — no API calls)
3. Toggle `RUN_*` flags per section (`False` = skip API calls)
4. After live cells, open **`docs/eval-runs/notebook-latest.json`**

**First time only** (terminal, before opening the notebook):
```bash
cd llm-log-triage && python -m venv .venv && source .venv/bin/activate
pip install -e ".[dev,obs]"
```

| Flag | Section | API calls | Report `section` |
|------|---------|----------|------------------|
| `RUN_GOOD` | Normal golden case | 1 triage | `good_case` |
| `RUN_ADV` | Adversarial case | 1 triage | `adversarial` |
| `RUN_CUSTOM` | Your own log text | 1 triage | `custom` |
| `RUN_JUDGE` | Triage + judge | 2 | `judge` |
| `RUN_JUDGE_FAIL` | Judge only (bad JSON injected) | 1 judge | `judge_fail` |
| `RUN_BATCH` | First N normal cases | N triage | `batch` |

**Section 8 (after 2–7):** **8A** review notebook report · **8B** pytest gates · **8C** LangSmith dataset + experiment

## Setup

Run this cell first after picking the `.venv` kernel.

In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

here = Path.cwd().resolve()
if (here / "src" / "llm_log_triage" / "chain.py").exists():
    ROOT = here
elif (here.parent / "src" / "llm_log_triage" / "chain.py").exists():
    ROOT = here.parent
else:
    raise RuntimeError(
        f"Cannot find repo root from {here}. Open from llm-log-triage/ or notebooks/."
    )
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from llm_log_triage.eval_checks import case_by_id, split_cases
from llm_log_triage.notebook_display import (
    show_case_header,
    show_expected,
    show_log,
    show_result,
)
from llm_log_triage.notebook_runner import (
    eval_golden_case,
    eval_with_judge,
    reset_session,
    run_batch,
    run_custom_log,
    run_judge_fail_proof,
)
from llm_log_triage.notebook_setup import load_project_env, verify_notebook_kernel
from llm_log_triage.schema import Category, Severity, TriageOutput

load_project_env(ROOT)
verify_notebook_kernel(ROOT)
os.chdir(ROOT)

reset_session()

GOLDEN_PATH = ROOT / "data" / "golden_set.json"
CASES = json.loads(GOLDEN_PATH.read_text())["cases"]
NORMAL_CASES, ADVERSARIAL_CASES = split_cases(CASES)

print(f"Repo: {ROOT}")
print(f"Golden set: {len(CASES)} cases ({len(NORMAL_CASES)} normal, {len(ADVERSARIAL_CASES)} adversarial)")
print(f"Report: {ROOT / 'docs/eval-runs/notebook-latest.json'}")

## 1. Browse golden set (free)

Inspect labels before spending API credits. Same file pytest uses.

In [ ]:
GOOD_CASE_ID = "gs-001"   # payments-api postgres connection refused
ADV_CASE_ID = "adv-002"   # garbage log text

good_case = case_by_id(CASES, GOOD_CASE_ID)
adv_case = case_by_id(CASES, ADV_CASE_ID)

display(Markdown(f"**Normal cases:** {[c['id'] for c in NORMAL_CASES[:5]]} … ({len(NORMAL_CASES)} total)"))
display(Markdown(f"**Adversarial:** {[c['id'] for c in ADVERSARIAL_CASES]}"))

show_case_header(good_case)
show_log(good_case)
show_expected(good_case)

## 2. Good case — triage + deterministic eval

Charter checks: schema, severity, category, must_mention. Mirrors `test_golden_set_pass_rate` for **one** row.

In [ ]:
RUN_GOOD = True  # set True for one live triage call

if RUN_GOOD:
    if not os.getenv("OPENAI_API_KEY") and not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env")
    show_case_header(good_case)
    show_log(good_case)
    show_expected(good_case)
    result, checks, report_path = eval_golden_case(good_case, section="good_case")
    show_result(result, checks)
    print(f"Report updated: {report_path}")
else:
    print(f"Skipped (RUN_GOOD=False). Will run {GOOD_CASE_ID} when enabled.")

## 3. Adversarial case — unknown + low confidence

Empty/garbage/truncated logs should return `severity=unknown`, `category=unknown`, low confidence. Mirrors `test_adversarial_cases`.

In [ ]:
RUN_ADV = True

if RUN_ADV:
    if not os.getenv("OPENAI_API_KEY") and not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env")
    show_case_header(adv_case)
    show_log(adv_case)
    show_expected(adv_case)
    result, checks, report_path = eval_golden_case(adv_case, section="adversarial")
    show_result(result, checks)
    print(f"Report updated: {report_path}")
else:
    print(f"Skipped (RUN_ADV=False). Will run {ADV_CASE_ID} when enabled.")

## 4. Custom log (ad-hoc)

Try any log — no golden label. Useful for Streamlit 👎 follow-up or new golden-set candidates.

In [ ]:
RUN_CUSTOM = True

CUSTOM_LOG = "2026-05-28T14:02:11Z ERROR auth-api connection refused postgres"
CUSTOM_SERVICE = "auth-api"

if RUN_CUSTOM:
    if not os.getenv("OPENAI_API_KEY") and not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env")
    display(Markdown(f"**Custom log:**\n```\n{CUSTOM_LOG}\n```"))
    result, report_path = run_custom_log(CUSTOM_LOG, CUSTOM_SERVICE, section="custom")
    show_result(result)
    print(f"Report updated: {report_path}")
else:
    print("Skipped (RUN_CUSTOM=False). Edit CUSTOM_LOG then set RUN_CUSTOM=True.")

## 5. LLM-as-Judge — one good case

Two LLM calls: triage → reviewer scores coherence + actionability (≥4/5 pass). Mirrors `test_judge_single_sample` / one row of `test_judge_pass_rate`.

LangSmith: filter `role:judge` and `case_id:gs-001`.

In [ ]:
RUN_JUDGE = True
JUDGE_CASE_ID = "gs-001"

if RUN_JUDGE:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Judge path needs OPENAI_API_KEY in .env")
    judge_case = case_by_id(CASES, JUDGE_CASE_ID)
    show_case_header(judge_case)
    result, det_checks, score, judge_ok, report_path = eval_with_judge(
        judge_case, section="judge"
    )
    show_result(result, det_checks)
    display(Markdown(
        f"**Judge scores:** coherence={score.coherence}, actionability={score.actionability}\n\n"
        f"Rationale: {score.rationale}\n\n"
        f"**Judge pass:** {'✅' if judge_ok else '❌'}"
    ))
    print(f"Report updated: {report_path}")
else:
    print(f"Skipped (RUN_JUDGE=False). Will triage + judge {JUDGE_CASE_ID} when enabled.")

## 6. Judge fail-proof — reviewer rejects bad triage

No triage call — injects contradictory JSON. Mirrors `test_judge_rejects_deliberately_bad_triage`. Recorded in `notebook-latest.json` under `section: judge_fail`.

In [ ]:
RUN_JUDGE_FAIL = True

if RUN_JUDGE_FAIL:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Judge path needs OPENAI_API_KEY in .env")
    log = good_case["log_text"]
    bad = TriageOutput(
        severity=Severity.SEV4,
        category=Category.UNKNOWN,
        likely_cause="No issue here; its healthy.",
        suggested_action="No action required.",
        confidence=0.1,
        evidence_lines=[],
    )
    display(Markdown("**Injected bad triage** (contradicts ERROR log):"))
    display(Markdown(f"```json\n{bad.model_dump()}\n```"))
    score, judge_ok, report_path = run_judge_fail_proof(log, bad, section="judge_fail")
    display(Markdown(
        f"**Judge scores:** coherence={score.coherence}, actionability={score.actionability}\n\n"
        f"Rationale: {score.rationale}\n\n"
        f"**Expected reject:** {'✅ rejected' if not judge_ok else '❌ unexpectedly passed'}"
    ))
    print(f"Report updated: {report_path}")
else:
    print("Skipped (RUN_JUDGE_FAIL=False).")

## 7. Batch sample — mini pass rate

Run first N **normal** cases and compute pass rate. Mirrors a slice of `test_golden_set_pass_rate` — use pytest for the full 22-case gate.

In [ ]:
RUN_BATCH = True
BATCH_N = 5

if RUN_BATCH:
    if not os.getenv("OPENAI_API_KEY") and not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env")
    rows, summary, report_path = run_batch(NORMAL_CASES[:BATCH_N], section="batch")
    display(Markdown(
        f"**Batch pass rate:** {summary['cases_passed']}/{summary['cases_run']} "
        f"({summary['pass_rate']:.0%})"
    ))
    for r in rows:
        icon = "✅" if r["passed"] else "❌"
        print(f"{icon} {r['id']} {r['checks']}")
    print(f"Report updated: {report_path}")
else:
    print(f"Skipped (RUN_BATCH=False). Will run first {BATCH_N} normal cases when enabled.")

## 8. Close the loop — reports, CI, LangSmith

Sections 2–7 are **interactive** (`interface:notebook`). Section 8 compares three **automation layers**:

| Step | What you are doing | Results land in |
|------|-------------------|-----------------|
| **8A** | Read back this notebook session | `docs/eval-runs/notebook-latest.json` |
| **8B** | Re-run pytest merge gates locally | terminal + `judge-*.json` (pytest only) |
| **8C** | Sync golden set → LangSmith dataset + run experiment | LangSmith UI + `langsmith-eval-latest.json` |

**Tracing note:** sections 2–7 also emit traces when `OBS_BACKEND=langsmith` (filter `interface:notebook`). **8C** is the full offline experiment on the golden set — same scorers as pytest, visible under **LangSmith → Datasets & Experiments**.

### 8A — Review notebook session report (free)

**Purpose:** confirm what sections 2–7 recorded in this kernel session.

**What it checks:**
- One row per live cell you ran (`good_case`, `adversarial`, `custom`, `judge`, …)
- Deterministic pass/fail vs golden labels (same `eval_checks` as pytest)
- Judge scores when section 5/6 ran

**What it does *not* do:** no LLM calls, no LangSmith upload, no CI gate.

Run after sections 2–7. Also lists other `*-latest.json` reports (pytest, LangSmith) if present.

In [ ]:
import json

report_path = ROOT / "docs/eval-runs/notebook-latest.json"
eval_dir = ROOT / "docs/eval-runs"

if report_path.exists():
    report = json.loads(report_path.read_text())
    display(Markdown(f"**Notebook report:** `{report_path}`"))
    display(Markdown(
        f"- Session started: `{report.get('session_started', '—')}`\n"
        f"- Runs: **{report.get('runs_count', len(report.get('runs', [])))}**\n"
        f"- Deterministic: {report.get('deterministic_passed', '?')}/{report.get('deterministic_scored', '?')} passed\n"
        f"- Judge: {report.get('judge_passed', '?')}/{report.get('judge_scored', '?')} passed"
    ))
    for run in report.get("runs", []):
        det = run.get("deterministic_checks", {})
        judge = run.get("judge")
        det_icon = "✅" if det.get("passed") else ("❌" if det else "—")
        judge_icon = "✅" if judge and judge.get("passed") else ("❌" if judge else "—")
        print(f"{det_icon} {run.get('section')} {run.get('case_id')} det={det_icon} judge={judge_icon}")
else:
    print("No report yet — run sections 2–7 with RUN_* = True, then re-run 8A.")

display(Markdown("**Other latest reports (pytest / LangSmith):**"))
for p in sorted(eval_dir.glob("*-latest.json")):
    if p.name == "notebook-latest.json":
        continue
    print(f"  {p.relative_to(ROOT)}")
    if p.name == "langsmith-eval-latest.json":
        ls = json.loads(p.read_text())
        display(Markdown(
            f"  → experiment **`{ls.get('experiment_name')}`** · dataset `{ls.get('dataset')}` · "
            f"project `{ls.get('project')}` · {ls.get('ui_hint', '')}"
        ))


### 8B — Run pytest CI gates (optional)

**Purpose:** run the same **automated** tests GitHub Actions uses (`interface:pytest`).

| Flag | Command | API cost | What it validates |
|------|---------|----------|-------------------|
| `RUN_PYTEST_FREE` | `pytest -m "not llm"` | None | Schema, eval_checks, notebook wiring |
| `RUN_PYTEST_LLM` | `pytest -m llm` | ~26 calls | Full golden set + adversarial (≥90% gate) |
| `RUN_PYTEST_JUDGE` | `pytest -m judge` | Extra reviewer calls | LLM-as-judge pass rate |

**When to run:** skip if you already passed these in terminal. Set **one flag at a time**.

**Outputs:** `judge-latest.json`, `judge-fail-smoke-latest.json` under `docs/eval-runs/`.

**Tip:** uses `.venv/bin/pytest` when present (avoid conda pytest on PATH).

In [ ]:
import subprocess

RUN_PYTEST_FREE = False   # pytest -m "not llm"
RUN_PYTEST_LLM = False    # pytest -m llm (costs $)
RUN_PYTEST_JUDGE = True  # pytest -m judge

pytest_bin = str(ROOT / ".venv" / "bin" / "pytest")
if not Path(pytest_bin).exists():
    pytest_bin = "pytest"

commands = []
if RUN_PYTEST_FREE:
    commands.append([pytest_bin, "tests/", "-m", "not llm", "-v"])
if RUN_PYTEST_LLM:
    commands.append([pytest_bin, "tests/", "-m", "llm", "-v"])
if RUN_PYTEST_JUDGE:
    commands.append([pytest_bin, "tests/", "-m", "judge", "-v", "-s"])

if not commands:
    print("All RUN_PYTEST_* flags are False. Terminal equivalents:")
    print(f"  {pytest_bin} tests/ -m 'not llm' -v")
    print(f"  {pytest_bin} tests/ -m llm -v")
    print(f"  {pytest_bin} tests/ -m judge -v -s")
else:
    for cmd in commands:
        print("\n$ " + " ".join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=False)


### 8C — LangSmith dataset + experiment (optional)

**Purpose:** push the golden set to a LangSmith **dataset**, then run an **experiment** over it.

**What gets uploaded:** inputs (log text), reference labels (`expected`), metadata — not your Python source.

**What runs locally:** `chain.invoke()` + the same `eval_checks` scorers as pytest (via LangSmith `evaluate()`).

**Requires:** `LANGCHAIN_API_KEY` in `.env`, `OBS_BACKEND=langsmith` (default).

| Flag | Action |
|------|--------|
| `RUN_LANGSMITH_SYNC` | Upload/update `log-triage-golden-set-v3` from `data/golden_set.json` |
| `RUN_LANGSMITH_EVAL` | Run experiment (auto-syncs dataset unless you synced in same cell) |

**Where to look:** LangSmith → project **`llm-log-triage`** → **Datasets & Experiments**. Local pointer: `docs/eval-runs/langsmith-eval-latest.json`.

Terminal equivalent: `./scripts/run_langsmith_eval_golden.sh`

In [ ]:
import os

RUN_LANGSMITH_SYNC = False   # upload golden set → LangSmith dataset
RUN_LANGSMITH_EVAL = True   # run experiment (costs $; uses OPENAI/ANTHROPIC + LANGCHAIN keys)
LANGSMITH_MAX_CASES = 5    # int or "all" for full 26-case run
LANGSMITH_DATASET = os.getenv("LOG_TRIAGE_LANGSMITH_DATASET", "log-triage-golden-set-v3")
LANGSMITH_PREFIX = os.getenv("LOG_TRIAGE_LANGSMITH_EXPERIMENT_PREFIX", "log-triage-v3")

if RUN_LANGSMITH_SYNC or RUN_LANGSMITH_EVAL:
    if not os.getenv("LANGCHAIN_API_KEY"):
        raise RuntimeError("Set LANGCHAIN_API_KEY in .env for LangSmith dataset/experiment")
    from llm_log_triage.langsmith_eval import (
        load_golden_cases,
        run_evaluation,
        sync_dataset,
        write_eval_report,
    )

    cases = load_golden_cases()
    if RUN_LANGSMITH_SYNC:
        name = sync_dataset(cases, dataset_name=LANGSMITH_DATASET)
        display(Markdown(f"**Dataset synced:** `{name}` ({len(cases)} examples)"))

    if RUN_LANGSMITH_EVAL:
        if not RUN_LANGSMITH_SYNC:
            sync_dataset(cases, dataset_name=LANGSMITH_DATASET)
            print(f"Dataset auto-synced: {LANGSMITH_DATASET} ({len(cases)} examples)")
        limit = None if str(LANGSMITH_MAX_CASES).lower() == "all" else int(LANGSMITH_MAX_CASES)
        os.environ.setdefault("OBS_BACKEND", "langsmith")
        summary = run_evaluation(
            dataset_name=LANGSMITH_DATASET,
            max_cases=limit,
            experiment_prefix=LANGSMITH_PREFIX,
        )
        report_path = write_eval_report(summary)
        display(Markdown(
            f"**Experiment:** `{summary['experiment_name']}`\n\n"
            f"- Dataset: `{summary['dataset']}`\n"
            f"- Cases: `{summary.get('max_cases', 'all')}`\n"
            f"- Project: `{summary['project']}`\n"
            f"- Report: `{report_path}`\n\n"
            f"Open LangSmith → Datasets & Experiments → find **`{summary['experiment_name']}`**"
        ))
else:
    print("Skipped (RUN_LANGSMITH_SYNC and RUN_LANGSMITH_EVAL are False).")
    print("Or run: ./scripts/run_langsmith_eval_golden.sh")
